# Scrape full job details for each job

For each job (you can go by URL), please try to get the following pieces of information:
1. Job Description
2. Do they ask additional questions?
3. Company Information (May not exist for some firms)
• Company name

• Industry

• Firm size

• Company description

• Perks and benefits

• Average rating

• Number of reviews

• Any other information

Please get this information for the most recent 1000 jobs and report how long the
scraping takes

# Selenium

In [12]:
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver

# Set up the driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

def get_job_urls_from_page(page_num):
    URL = f"https://sg.jobstreet.com/jobs?page={page_num}&sortmode=ListedDate"
    driver.get(URL)
    time.sleep(3)  # Wait for the page to load

    job_urls = []
    job_cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="job-card"]')
    
    for job in job_cards:
        url_tag = job.find_element(By.CSS_SELECTOR, '[data-automation="job-list-view-job-link"]')
        job_url = url_tag.get_attribute('href') if url_tag else 'N/A'
        job_urls.append(job_url)
    
    return job_urls

# Get job URLs from multiple pages
job_urls = []
page_num = 1
while len(job_urls) < 20:
    job_urls.extend(get_job_urls_from_page(page_num))
    page_num += 1
    if len(job_urls) >= 20:
        break

# Remove duplicates (if any)
job_urls = list(set(job_urls))[:20]


In [13]:
len(job_urls)

20

In [ ]:
def get_job_details(job_url):
    driver.get(job_url)
    time.sleep(3)  # Wait for the job page to load

    job_details = {}

    job_details = job_url
    
    # Job Description
    try:
        description_tag = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-automation="jobAdDetails"]'))
        )
        job_details['Job Description'] = description_tag.text.strip() if description_tag else 'N/A'
    except Exception as e:
        job_details['Job Description'] = 'N/A'

    # Employer Questions
    try:
        questions_section = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, "//h2[contains(text(), 'Employer questions')]/following-sibling::div//ul"))
        )
        question_items = questions_section.find_elements(By.XPATH, ".//li")
        questions = [item.text.strip() for item in question_items if item.text.strip()]
        job_details['Employer Questions'] = questions if questions else ['N/A']
    except Exception as e:
        job_details['Employer Questions'] = ['N/A']
    
    return job_details

# Extract details for each job URL
job_data = []
for job_url in job_urls:
    try:
        job_details = get_job_details(job_url)
        job_data.append(job_details)
    except Exception as e:
        print(f"Error occurred while processing {job_url}: {e}")
        job_data.append({'Job Description': 'N/A', 'Employer Questions': 'N/A'})

# Create DataFrame
df = pd.DataFrame(job_data)

# Save to Excel
df.to_excel("20_job_details_with_questions.xlsx", index=False)
print("Job details saved to Excel file.")


Job details saved to Excel file.


In [14]:
def get_job_details(job_url):
    driver.get(job_url)
    time.sleep(3)  # Wait for the job page to load

    job_details = {}

    # Job URL to trace back
    job_details['Job URL'] = job_url

    # Job Description
    try:
        description_tag = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-automation="jobAdDetails"]'))
        )
        job_details['Job Description'] = description_tag.text.strip() if description_tag else 'N/A'
    except Exception as e:
        job_details['Job Description'] = 'N/A'

    # Employer Questions
    try:
        questions_section = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, "//h2[contains(text(), 'Employer questions')]/following-sibling::div//ul"))
        )
        question_items = questions_section.find_elements(By.XPATH, ".//li")
        questions = [item.text.strip() for item in question_items if item.text.strip()]
        job_details['Employer Questions'] = questions if questions else ['N/A']
    except Exception as e:
        job_details['Employer Questions'] = ['N/A']
    

    # Get Company Profile
    try:
        company_section = driver.find_element(By.CSS_SELECTOR, '[data-automation="company-profile"]')
        
        # Get Company Name
        try:
            company_name = company_section.find_element(By.CSS_SELECTOR, 'button h4')
            job_details["Company Name"] = company_name.text.strip()
        except Exception as e:
            job_details['Company Name'] = 'N/A'
        
        # Get Industry (flexible extraction)
        try:
            industry_tag = company_section.find_element(By.CSS_SELECTOR, 'span.gepq850.eihuid5b span.eihuid4z')
            job_details["Industry"] = industry_tag.text.strip() if industry_tag else 'N/A'
        except Exception as e:
            job_details['Industry'] = 'N/A'
        
        # Get Firm Size
        try:
            firm_size = company_section.find_element(By.XPATH, "//span[contains(text(),'employees')]")
            job_details["Firm Size"] = firm_size.text.strip() if firm_size else 'N/A'
        except Exception as e:
            job_details['Firm Size'] = 'N/A'
        
        # Get Company Description (more generic extraction)
        try:
            # Find the first span containing the first paragraph
            first_paragraph = company_section.find_element(By.CSS_SELECTOR, 'span.gepq850.eihuid4z p.eihuidcz')
    
            # Find the second span containing the second paragraph
            second_paragraph = company_section.find_element(By.CSS_SELECTOR, 'span.gepq850.eihuid4z p.eihuidcb')
    
            # Combine the paragraphs (if both exist)
            company_description = f"{first_paragraph} {second_paragraph}".strip() if first_paragraph or second_paragraph else 'N/A'
    
            # Store the company description
            job_details["Company Description"] = company_description
        except Exception as e:
            job_details['Company Description'] = 'N/A'

        
        # Get Perks and Benefits
        try:
            # Find the element that contains "Perks and benefits" text
            perks_section = company_section.find_element(By.XPATH, "//span[contains(text(),'Perks and benefits')]/following-sibling::div")
    
            # Find all perk items inside this section
            perk_items = perks_section.find_elements(By.CSS_SELECTOR, "span.gepq850.eihuid4z i7p5ej0 i7p5ej1 i7p5ej21 .gepq850.wol1j10")
    
            # Extract and clean the text from each perk item
            perks_list = [perk.text.strip() for perk in perk_items if perk.text.strip()]
    
            # If perks exist, assign to the job details, else 'N/A'
            job_details["Perks and Benefits"] = perks_list if perks_list else 'N/A'
        except Exception as e:
            job_details['Perks and Benefits'] = 'N/A'


        
        # Get Average Rating
        try:
            avg_rating = company_section.find_element(By.CSS_SELECTOR, '[data-automation="company-profile-review-rating"]')
            job_details["Average Rating"] = avg_rating.text.strip() if avg_rating else 'N/A'
        except Exception as e:
            job_details['Average Rating'] = 'N/A'
        
        # Get Number of Reviews
        try:
            num_reviews = company_section.find_element(By.CSS_SELECTOR, '[data-automation="company-profile-review-link"]')
            job_details["Number of Reviews"] = num_reviews.text.strip() if num_reviews else 'N/A'
        except Exception as e:
            job_details['Number of Reviews'] = 'N/A'
        
    except Exception as e:
        job_details['Company Name'] = 'N/A'
        job_details['Industry'] = 'N/A'
        job_details['Firm Size'] = 'N/A'
        job_details['Company Description'] = 'N/A'
        job_details['Perks and Benefits'] = 'N/A'
        job_details['Average Rating'] = 'N/A'
        job_details['Number of Reviews'] = 'N/A'

    return job_details


# Extract details for each job URL
job_data = []
for job_url in job_urls:
    job_details = get_job_details(job_url)
    job_data.append(job_details)

# Create DataFrame
df = pd.DataFrame(job_data)


In [16]:
df.head(20)

,Job URL,Job Description,Employer Questions,Company Name,Industry,Firm Size,Company Description,Perks and Benefits,Average Rating,Number of Reviews
0,https://sg.jobstreet.com/job/83049189?type=sta...,FACULTY OPENING FOR SPEECH AND LANGUAGE THERAP...,[N/A],Singapore Institute of Technology,,"101-1,000 employees",<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.8,10 reviews
1,https://sg.jobstreet.com/job/83049285?type=sta...,The HR & Admin Assistant supports the HR and a...,[Which of the following statements best descri...,N/A,N/A,N/A,N/A,N/A,N/A,N/A
2,https://sg.jobstreet.com/job/83049309?type=sta...,"Key Responsibilities:\nProject planning, sched...",[N/A],Singapore Institute of Technology,,"101-1,000 employees",<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.8,10 reviews
3,https://sg.jobstreet.com/job/83049269?type=sta...,Key Responsibilities:\nParticipate in and mana...,[N/A],Singapore Institute of Technology,,"101-1,000 employees",<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.8,10 reviews
4,https://sg.jobstreet.com/job/83049187?type=sta...,About the role\n\nWe are seeking an experience...,[Which of the following statements best descri...,N/A,N/A,N/A,N/A,N/A,N/A,N/A
5,https://sg.jobstreet.com/job/83049236?type=sta...,Duration: 6 months\nLocation: One North\nWorki...,[Which of the following statements best descri...,Persolkelly,,51-100 employees,<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.7,7 reviews
6,https://sg.jobstreet.com/job/83049277?type=sta...,Key Responsibilities:\nParticipate in and mana...,[N/A],Singapore Institute of Technology,,"101-1,000 employees",<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.8,10 reviews
7,https://sg.jobstreet.com/job/83049168?type=sta...,We have a few openings for Research Fellow or ...,[N/A],Singapore Institute of Technology,,"101-1,000 employees",<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.8,10 reviews
8,https://sg.jobstreet.com/job/83049246?type=sta...,FACULTY OPENINGS AT THE SINGAPORE INSTITUTE OF...,[N/A],Singapore Institute of Technology,,"101-1,000 employees",<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.8,10 reviews
9,https://sg.jobstreet.com/job/83049251?type=sta...,FACULTY OPENINGS AT THE SINGAPORE INSTITUTE OF...,[N/A],Singapore Institute of Technology,,"101-1,000 employees",<selenium.webdriver.remote.webelement.WebEleme...,N/A,2.8,10 reviews


In [17]:
driver.quit()

37m 10.2s for 200 jobs with job description and additional questions only

3m 26s for 20 jobs with job description and additional questions only

6m 51s for 20 jobs with all job details (with errors)